# Entrenamiento y evaluación reproducible - GRU + Log Mel-Spectrogram

Este notebook permite reproducir el entrenamiento del modelo final utilizado en el proyecto de domótica controlada por voz.

El flujo implementado es:

1. Cargar el corpus propio desde carpetas por clase.
2. Dividir el dataset en entrenamiento, validación y prueba.
3. Convertir cada audio a **Log Mel-Spectrogram**.
4. Entrenar un modelo secuencial **GRU + Mel**.
5. Generar curvas de entrenamiento.
6. Calcular matriz de confusión y métricas: accuracy, precision, recall y F1-score.
7. Guardar los resultados en archivos reutilizables para el reporte.

> Este notebook asume la misma estructura del proyecto usada en los scripts de entrenamiento.


## 0. Instalación de dependencias

Se instalan todas las librerías necesarias desde el archivo `requirements.txt`.


In [ ]:
%pip install -r requirements.txt -q
print("✓ Todas las dependencias se han instalado correctamente.")


## 1. Importación de librerías

Se importan las librerías necesarias para manejo de datos, audio, entrenamiento del modelo, gráficas y métricas.


In [4]:
import json
import csv
from pathlib import Path

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    accuracy_score
)

print("TensorFlow:", tf.__version__)


ModuleNotFoundError: No module named 'numpy'

## 2. Configuración del proyecto

Si el archivo `config.py` ya existe en el proyecto, el notebook intentará usar sus valores. Si no existe, se definen valores por defecto que pueden ajustarse manualmente.


In [ ]:
try:
    from config import (
        DATASET_DIR,
        MODELS_DIR,
        COMMANDS,
        SAMPLE_RATE,
        AUDIO_SAMPLES,
        FRAME_LENGTH,
        FRAME_STEP,
        BATCH_SIZE,
        EPOCHS,
        VALIDATION_SPLIT,
        TEST_SPLIT,
        RANDOM_SEED,
        MEL_BINS,
        LOWER_EDGE_HERTZ,
        UPPER_EDGE_HERTZ,
    )
    print("Configuración cargada desde config.py")
except Exception as e:
    print("No se pudo importar config.py. Usando configuración por defecto.")
    print(e)

    DATASET_DIR = Path("dataset_augmented")
    MODELS_DIR = Path("models")

    COMMANDS = {
        "LUCES_ON": "enciende las luces",
        "LUCES_OFF": "apaga las luces",
        "AIRE_ON": "enciende el aire",
        "AIRE_OFF": "apaga el aire",
        "BOMBA_REGAR": "riega las plantas",
        "RUIDO_FONDO": "ruido fondo",
    }

    SAMPLE_RATE = 16000
    AUDIO_SAMPLES = 16000 * 2
    FRAME_LENGTH = 256
    FRAME_STEP = 128
    BATCH_SIZE = 32
    EPOCHS = 30
    VALIDATION_SPLIT = 0.15
    TEST_SPLIT = 0.15
    RANDOM_SEED = 42

    MEL_BINS = 40
    LOWER_EDGE_HERTZ = 80.0
    UPPER_EDGE_HERTZ = 7600.0

DATASET_DIR = Path(DATASET_DIR)
MODELS_DIR = Path(MODELS_DIR)
MODELS_DIR.mkdir(exist_ok=True)

tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("Dataset:", DATASET_DIR)
print("Modelos:", MODELS_DIR)
print("Clases:", list(COMMANDS.keys()))
print("Sample rate:", SAMPLE_RATE)
print("Audio samples:", AUDIO_SAMPLES)
print("Mel bins:", MEL_BINS)


## 3. Construcción de la lista de archivos

El dataset debe estar organizado en carpetas, una por cada clase:

```text
dataset_augmented/
├── LUCES_ON/
├── LUCES_OFF/
├── AIRE_ON/
├── AIRE_OFF/
├── BOMBA_REGAR/
└── RUIDO_FONDO/
```


In [ ]:
def construir_lista_archivos():
    nombres_clases = list(COMMANDS.keys())
    archivos = []
    etiquetas = []

    for indice, clase in enumerate(nombres_clases):
        carpeta_clase = DATASET_DIR / clase

        if not carpeta_clase.exists():
            print(f"Advertencia: no existe la carpeta {carpeta_clase}")
            continue

        archivos_wav = sorted(carpeta_clase.glob("*.wav"))

        for archivo in archivos_wav:
            archivos.append(str(archivo))
            etiquetas.append(indice)

    archivos = np.array(archivos)
    etiquetas = np.array(etiquetas)

    if len(archivos) == 0:
        raise RuntimeError("No se encontraron archivos .wav en el dataset.")

    return archivos, etiquetas, nombres_clases


archivos, etiquetas, nombres_clases = construir_lista_archivos()

print("Total de audios encontrados:", len(archivos))
print("\nDistribución por clase:")

for i, clase in enumerate(nombres_clases):
    cantidad = np.sum(etiquetas == i)
    porcentaje = cantidad / len(etiquetas) * 100
    print(f"{i:02d}. {clase:20s} -> {cantidad:5d} audios | {porcentaje:6.2f}%")


## 4. División del dataset

Se separa el dataset en entrenamiento, validación y prueba. Se usa `stratify` para conservar la proporción de clases en cada conjunto.


In [ ]:
def dividir_dataset(archivos, etiquetas):
    test_size_total = TEST_SPLIT + VALIDATION_SPLIT

    archivos_train, archivos_temp, etiquetas_train, etiquetas_temp = train_test_split(
        archivos,
        etiquetas,
        test_size=test_size_total,
        random_state=RANDOM_SEED,
        stratify=etiquetas
    )

    proporcion_validacion = VALIDATION_SPLIT / test_size_total

    archivos_val, archivos_test, etiquetas_val, etiquetas_test = train_test_split(
        archivos_temp,
        etiquetas_temp,
        test_size=1 - proporcion_validacion,
        random_state=RANDOM_SEED,
        stratify=etiquetas_temp
    )

    return (
        archivos_train,
        etiquetas_train,
        archivos_val,
        etiquetas_val,
        archivos_test,
        etiquetas_test
    )


(
    archivos_train,
    etiquetas_train,
    archivos_val,
    etiquetas_val,
    archivos_test,
    etiquetas_test
) = dividir_dataset(archivos, etiquetas)

print("Entrenamiento:", len(archivos_train))
print("Validación:   ", len(archivos_val))
print("Prueba:       ", len(archivos_test))


## 5. Preprocesamiento: audio a Log Mel-Spectrogram

Cada audio se convierte a mono, se ajusta a duración fija y luego se transforma a Log Mel-Spectrogram.

La salida tiene forma:

```text
tiempo × 40 bandas Mel
```

Esta secuencia es la entrada del modelo GRU.


In [ ]:
def cargar_audio(ruta_archivo):
    audio_binario = tf.io.read_file(ruta_archivo)

    audio, _ = tf.audio.decode_wav(
        audio_binario,
        desired_channels=1
    )

    audio = tf.squeeze(audio, axis=-1)
    longitud_audio = tf.shape(audio)[0]

    def rellenar():
        faltante = AUDIO_SAMPLES - longitud_audio
        return tf.pad(audio, paddings=[[0, faltante]])

    def recortar():
        return audio[:AUDIO_SAMPLES]

    audio = tf.cond(
        longitud_audio < AUDIO_SAMPLES,
        rellenar,
        recortar
    )

    return audio


def convertir_a_log_mel_spectrogram(audio):
    stft = tf.signal.stft(
        audio,
        frame_length=FRAME_LENGTH,
        frame_step=FRAME_STEP
    )

    spectrogram = tf.abs(stft)
    num_spectrogram_bins = FRAME_LENGTH // 2 + 1

    mel_weight_matrix = tf.signal.linear_to_mel_weight_matrix(
        num_mel_bins=MEL_BINS,
        num_spectrogram_bins=num_spectrogram_bins,
        sample_rate=SAMPLE_RATE,
        lower_edge_hertz=LOWER_EDGE_HERTZ,
        upper_edge_hertz=UPPER_EDGE_HERTZ
    )

    mel_spectrogram = tf.matmul(spectrogram, mel_weight_matrix)
    log_mel_spectrogram = tf.math.log(mel_spectrogram + 1e-6)

    media = tf.reduce_mean(log_mel_spectrogram)
    desviacion = tf.math.reduce_std(log_mel_spectrogram)

    log_mel_spectrogram = (
        log_mel_spectrogram - media
    ) / (desviacion + 1e-6)

    return log_mel_spectrogram


def preprocesar_audio(ruta_archivo, etiqueta):
    audio = cargar_audio(ruta_archivo)
    log_mel = convertir_a_log_mel_spectrogram(audio)
    return log_mel, etiqueta


def crear_tf_dataset(archivos, etiquetas, mezclar=False):
    dataset = tf.data.Dataset.from_tensor_slices((archivos, etiquetas))

    if mezclar:
        dataset = dataset.shuffle(
            buffer_size=len(archivos),
            seed=RANDOM_SEED,
            reshuffle_each_iteration=True
        )

    dataset = dataset.map(
        preprocesar_audio,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.cache()
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset


dataset_train = crear_tf_dataset(archivos_train, etiquetas_train, mezclar=True)
dataset_val = crear_tf_dataset(archivos_val, etiquetas_val, mezclar=False)
dataset_test = crear_tf_dataset(archivos_test, etiquetas_test, mezclar=False)

for batch_x, batch_y in dataset_train.take(1):
    input_shape = batch_x.shape[1:]
    print("Forma de entrada:", input_shape)
    print("Forma de etiquetas:", batch_y.shape)


## 6. Visualización de un Log Mel-Spectrogram

Esta celda permite visualizar una muestra ya transformada a Log Mel-Spectrogram.


In [ ]:
for muestra, etiqueta in dataset_train.take(1):
    ejemplo = muestra[0].numpy().T
    clase = nombres_clases[int(etiqueta[0].numpy())]

    plt.figure(figsize=(10, 4))
    plt.imshow(ejemplo, aspect="auto", origin="lower")
    plt.title(f"Ejemplo Log Mel-Spectrogram - {clase}")
    plt.xlabel("Tiempo")
    plt.ylabel("Bandas Mel")
    plt.colorbar(label="Energía normalizada")
    plt.tight_layout()
    plt.show()
    break


## 7. Definición del modelo GRU + Mel

El modelo final utiliza una GRU simple, no bidireccional. La salida usa Softmax porque el sistema clasifica una sola clase por fragmento de voz.


In [ ]:
def crear_modelo_gru_mel(input_shape, numero_clases):
    modelo = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),

        tf.keras.layers.LayerNormalization(),

        tf.keras.layers.GRU(
            96,
            return_sequences=True,
            dropout=0.25
        ),

        tf.keras.layers.GRU(
            64,
            return_sequences=False,
            dropout=0.25
        ),

        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dropout(0.35),

        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dropout(0.25),

        tf.keras.layers.Dense(numero_clases, activation="softmax")
    ])

    modelo.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return modelo


modelo = crear_modelo_gru_mel(
    input_shape=input_shape,
    numero_clases=len(nombres_clases)
)

modelo.summary()


## 8. Entrenamiento del modelo

Se utiliza `EarlyStopping` para evitar sobreentrenamiento y `ModelCheckpoint` para guardar el mejor modelo según la exactitud de validación.


In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=7,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(MODELS_DIR / "domotica_gru_mel_notebook.keras"),
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    )
]

historial = modelo.fit(
    dataset_train,
    validation_data=dataset_val,
    epochs=EPOCHS,
    callbacks=callbacks
)


## 9. Guardado del modelo y clases

Se guarda el modelo entrenado y el archivo de clases utilizado para interpretar las predicciones.


In [ ]:
modelo.save(MODELS_DIR / "domotica_gru_mel_notebook.keras")

with open(MODELS_DIR / "class_names_gru_mel_notebook.json", "w", encoding="utf-8") as archivo_json:
    json.dump(nombres_clases, archivo_json, indent=2, ensure_ascii=False)

print("Modelo guardado en:", MODELS_DIR / "domotica_gru_mel_notebook.keras")
print("Clases guardadas en:", MODELS_DIR / "class_names_gru_mel_notebook.json")


## 10. Curvas de entrenamiento

Se generan las gráficas de exactitud y pérdida para observar el comportamiento del modelo durante el entrenamiento.


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(historial.history["accuracy"], label="Entrenamiento")
plt.plot(historial.history["val_accuracy"], label="Validación")
plt.title("Exactitud durante entrenamiento - GRU + Log Mel")
plt.xlabel("Época")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.savefig(MODELS_DIR / "training_accuracy_gru_mel_notebook.png", dpi=160, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(historial.history["loss"], label="Entrenamiento")
plt.plot(historial.history["val_loss"], label="Validación")
plt.title("Pérdida durante entrenamiento - GRU + Log Mel")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.savefig(MODELS_DIR / "training_loss_gru_mel_notebook.png", dpi=160, bbox_inches="tight")
plt.show()


## 11. Evaluación del modelo

Se evalúa el modelo sobre el conjunto de prueba. Aquí se calculan accuracy, precision, recall, F1-score y matriz de confusión.


In [ ]:
y_real = []
y_predicho = []

for x_batch, y_batch in dataset_test:
    predicciones = modelo.predict(x_batch, verbose=0)
    clases_predichas = np.argmax(predicciones, axis=1)

    y_real.extend(y_batch.numpy())
    y_predicho.extend(clases_predichas)

y_real = np.array(y_real)
y_predicho = np.array(y_predicho)

accuracy = accuracy_score(y_real, y_predicho)

reporte_texto = classification_report(
    y_real,
    y_predicho,
    target_names=nombres_clases,
    zero_division=0
)

reporte_dict = classification_report(
    y_real,
    y_predicho,
    target_names=nombres_clases,
    zero_division=0,
    output_dict=True
)

print("Accuracy general:", round(accuracy, 4))
print("\nReporte de clasificación:\n")
print(reporte_texto)


## 12. Matriz de confusión

La matriz permite observar los aciertos en la diagonal principal y las confusiones fuera de ella.


In [ ]:
matriz = confusion_matrix(
    y_real,
    y_predicho,
    labels=list(range(len(nombres_clases)))
)

figura, eje = plt.subplots(figsize=(10, 10))
display = ConfusionMatrixDisplay(
    confusion_matrix=matriz,
    display_labels=nombres_clases
)
display.plot(
    ax=eje,
    xticks_rotation=90,
    cmap="Blues",
    colorbar=False
)

plt.title("Matriz de confusión - GRU + Log Mel-Spectrogram")
plt.savefig(MODELS_DIR / "confusion_matrix_gru_mel_notebook.png", dpi=160, bbox_inches="tight")
plt.show()


## 13. Guardado de métricas

Se guardan los resultados para utilizarlos directamente en el reporte escrito.


In [ ]:
ruta_reporte_txt = MODELS_DIR / "classification_report_gru_mel_notebook.txt"
ruta_reporte_csv = MODELS_DIR / "classification_report_gru_mel_notebook.csv"
ruta_matriz_csv = MODELS_DIR / "confusion_matrix_gru_mel_notebook.csv"

with open(ruta_reporte_txt, "w", encoding="utf-8") as archivo:
    archivo.write("=== REPORTE DE CLASIFICACIÓN - GRU + LOG MEL-SPECTROGRAM ===\n\n")
    archivo.write(f"Accuracy general: {accuracy:.4f}\n\n")
    archivo.write(reporte_texto)

with open(ruta_reporte_csv, "w", newline="", encoding="utf-8") as archivo_csv:
    writer = csv.writer(archivo_csv)
    writer.writerow(["clase", "precision", "recall", "f1_score", "support"])

    for clase, metricas in reporte_dict.items():
        if isinstance(metricas, dict):
            writer.writerow([
                clase,
                metricas.get("precision", ""),
                metricas.get("recall", ""),
                metricas.get("f1-score", ""),
                metricas.get("support", "")
            ])

np.savetxt(
    ruta_matriz_csv,
    matriz,
    delimiter=",",
    fmt="%d"
)

print("Archivos generados:")
print("-", ruta_reporte_txt)
print("-", ruta_reporte_csv)
print("-", ruta_matriz_csv)
print("-", MODELS_DIR / "confusion_matrix_gru_mel_notebook.png")
print("-", MODELS_DIR / "training_accuracy_gru_mel_notebook.png")
print("-", MODELS_DIR / "training_loss_gru_mel_notebook.png")


## 14. Interpretación breve de resultados

Esta celda genera una interpretación base a partir de las métricas obtenidas. Puede adaptarse para el informe final.


In [ ]:
print("Interpretación automática:")
print()
print(f"El modelo GRU + Log Mel-Spectrogram obtuvo un accuracy general de {accuracy:.4f}.")
print("Esto indica que el modelo clasificó correctamente la mayoría de muestras del conjunto de prueba.")
print("Las métricas por clase permiten identificar si alguna categoría presenta menor precisión, recall o F1-score.")
print("La matriz de confusión permite observar en qué clases se concentran los errores y si existen falsos positivos hacia comandos válidos.")
